In [1]:
# Libraries
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Layer, LeakyReLU, ReLU, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.initializers import HeNormal, GlorotNormal
from tensorflow.keras.callbacks import LearningRateScheduler
from tensorflow.keras.regularizers import l1_l2
import numpy as np

from keras.regularizers import l1, l2
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import KFold

from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error, mean_absolute_error

In [17]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler
import pandas as pd
import json
# Import config variable
with open("../../config.json") as f:
    cfg = json.load(f)
    
target = cfg["rf"]        
file_name = cfg["rf_file_name"]   
random_state = cfg["split"]["random_state"]  
# Load the dataset
data = pd.read_csv(f"../../data/{file_name}.csv")
# data = pd.read_csv(f"../../data/dataset_dropna.csv")

In [18]:
X = data.drop(columns=[target])
y = data[target]
X_scaler = MinMaxScaler()
y_scaler = MinMaxScaler()

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Tirage de 4 seeds aléatoires différentes
master_rng = np.random.default_rng(seed=99)
seeds = master_rng.choice(10000, size=4, replace=False).tolist()
print("Seeds sélectionnées :", seeds)

# Split fixe (random_state=42) utilisé par la grille de recherche ci-dessous
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
X_test_scaled  = pd.DataFrame(scaler.transform(X_test),      columns=X_test.columns)

In [20]:
from tensorflow.keras.initializers import HeNormal, GlorotNormal

k_folds_train = 5
kf = KFold(n_splits=k_folds_train, shuffle=True, random_state=42)

initGlorotNormal = tf.keras.initializers.GlorotNormal()
l1_strength = 0.001
l2_strength = 0.01

early_stopping = EarlyStopping(patience=20, restore_best_weights=True)

mse_scores_train = []
mse_scores_val = []


r_squareds_train_all = []

r_squareds_val_all = []

models = []
historys = []

r_batch = []
batch_sizes = [4, 7, 10, 13, 16]
# batch = 14
# lr = 0.001
learning_rates = [0.0003, 0.0005, 0.0008, 0.001, 0.002]

r_squareds_train_lr = []
r_squareds_val_lr = []
mse_scores_train_lr = []
mse_scores_val_lr = []

for lr in learning_rates:
    
    for batch in batch_sizes :
        r_squareds_train_batch = []
        r_squareds_val_batch = []
        mse_scores_train_batch = []
        mse_scores_val_batch = []

        ann_rf = Sequential([
        Dense(32, activation= 'relu', kernel_initializer=initGlorotNormal),
        Dense(16, activation= 'relu', kernel_initializer=initGlorotNormal),
        Dropout(0.2),
        Dense(1, activation='linear', kernel_regularizer= l2(l2_strength))
        ])

        optimizer = tf.keras.optimizers.Adam(learning_rate= lr, decay = 0.01) # , decay = 0.01

        ann_rf.compile(optimizer=optimizer, 
                loss= 'mse', 
                metrics=['mae'])

        for train_index, val_index in kf.split(X_train_scaled):
            X_train_fold, X_val_fold = X_train_scaled.iloc[train_index], X_train_scaled.iloc[val_index]
            y_train_fold, y_val_fold = y_train.iloc[train_index], y_train.iloc[val_index]

            # lr_scheduler_callback = LearningRateScheduler(lr_scheduler)

            history_train = ann_rf.fit(X_train_fold, y_train_fold, epochs=100, batch_size=batch, verbose=0)
            historys.append(history_train)

            y_pred_train = ann_rf.predict(X_train_fold)
            r_squared_train = r2_score(y_train_fold, y_pred_train)
            r_squareds_train_batch.append(r_squared_train)
            mse_train = mean_squared_error(y_train_fold, y_pred_train)
            mse_scores_train_batch.append(mse_train)

            y_pred_val = ann_rf.predict(X_val_fold)
            r_squared_val = r2_score(y_val_fold, y_pred_val)
            r_squareds_val_batch.append(r_squared_val)
            mse_val = mean_squared_error(y_val_fold, y_pred_val)
            mse_scores_val_batch.append(mse_val)

        r_squareds_train_lr.append(r_squareds_train_batch)
        r_squareds_val_lr.append(r_squareds_val_batch)
        mse_scores_train_lr.append(mse_scores_train_batch)
        mse_scores_val_lr.append(mse_scores_val_batch)
        models.append(ann_rf)
        print('Model indice :', len(models)-1)

        print('With Learning rate =', lr,"With batch =", batch, "R²_train =", np.mean(r_squareds_train_batch))
        print('With Learning rate =', lr,"With batch =", batch, "mse_train =", np.mean(mse_scores_train_batch))
        print('')
        print('With Learning rate =', lr,"With batch =", batch, "R²_val =", np.mean(r_squareds_val_batch))
        print('With Learning rate =', lr,"With batch =", batch, "mse_val =", np.mean(mse_scores_val_batch))
        print('')
        y_pred_test= ann_rf.predict(X_test_scaled)
        # y_pred_test = y_scaler.inverse_transform(y_pred_test_scaled)
        r2_rc = r2_score(y_test, y_pred_test)

        print('With Learning rate =', lr, 'With batch =', batch, "R-squared on Test Set:", r2_rc)
            #MSE
        mae_rc = mean_absolute_error(y_test, y_pred_test)

        mse_test = mean_squared_error(y_test, y_pred_test)

        print('With Learning rate =', lr,'With batch =', batch, 'Mean Absolute Error test:', mae_rc)
        print('With Learning rate =', lr,'With batch =', batch, 'Mean Squared Error test:', mse_test)
        print('With Learning rate =', lr,'With batch =', batch, 'Root Mean Squared Error test:', np.sqrt(mse_test))
        print('')
        print('With Learning rate =', lr,'With batch_size = ', batch, 'Act vs Pred :')

C:\Users\matte\AppData\Roaming\Python\Python313\site-packages\keras\src\optimizers\base_optimizer.py:86: UserWarning: Argument `decay` is no longer supported and will be ignored.
  warnings.warn(


8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step 
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step 
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
Model indice : 0
With Learning rate = 0.0003 With batch = 4 R²_train = 0.7580947399139404
With Learning rate = 0.0003 With batch = 4 mse_train = 0.7628090023994446

With Learning rate = 0.0003 With batch = 4 R²_val = 0.6284778594970704
With Learning rate = 0.0003 With batch = 4 mse_val = 1.1737646579742431

3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step
With Learning rate = 0.0003 With batch = 4 R-squared on Test Set: 0.7047265768051147
With Learning rate = 0.0003 With batch = 4 Mean Absolute Error test: 0.8210124969482422
With Learning rate = 0.0003 With batch = 4 Mean Squared Error test: 1.0192384719848633
With Learning

C:\Users\matte\AppData\Roaming\Python\Python313\site-packages\keras\src\optimizers\base_optimizer.py:86: UserWarning: Argument `decay` is no longer supported and will be ignored.
  warnings.warn(


8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step 
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
Model indice : 1
With Learning rate = 0.0003 With batch = 7 R²_train = 0.6633515954017639
With Learning rate = 0.0003 With batch = 7 mse_train = 1.0614352345466613

With Learning rate = 0.0003 With batch = 7 R²_val = 0.5416508197784424
With Learning rate = 0.0003 With batch = 7 mse_val = 1.4478543043136596

3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step 
With Learning rate = 0.0003 With batch = 7 R-squared on Test Set: 0.5766938328742981
With Learning rate = 0.0003 With batch = 7 Mean Absolute Error test: 1.0009790658950806
With Learning rate = 0.0003 With batch = 7 Mean Squared Error test: 1.4611878395080566
With Learning

C:\Users\matte\AppData\Roaming\Python\Python313\site-packages\keras\src\optimizers\base_optimizer.py:86: UserWarning: Argument `decay` is no longer supported and will be ignored.
  warnings.warn(


8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step 
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
Model indice : 2
With Learning rate = 0.0003 With batch = 10 R²_train = 0.6614544034004212
With Learning rate = 0.0003 With batch = 10 mse_train = 1.067118513584137

With Learning rate = 0.0003 With batch = 10 R²_val = 0.5438138604164123
With Learning rate = 0.0003 With batch = 10 mse_val = 1.4414467573165894

3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step 
With Learning rate = 0.0003 With batch = 10 R-squared on Test Set: 0.5485880374908447
With Learning rate = 0.0003 With batch = 10 Mean Absolute Error test: 1.06257164478302
With Learning rate = 0.0003 With batch = 10 Mean Squared Error test: 1.5582046508789062
With Lear

... (40 outputs left out)

C:\Users\matte\AppData\Roaming\Python\Python313\site-packages\keras\src\optimizers\base_optimizer.py:86: UserWarning: Argument `decay` is no longer supported and will be ignored.
  warnings.warn(


8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step 
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step 
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step
Model indice : 23
With Learning rate = 0.002 With batch = 13 R²_train = 0.8610435009002686
With Learning rate = 0.002 With batch = 13 mse_train = 0.43831230998039244

With Learning rate = 0.002 With batch = 13 R²_val = 0.729482889175415
With Learning rate = 0.002 With batch = 13 mse_val = 0.854104733467102

3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step
With Learning rate = 0.002 With batch = 13 R-squared on Test Set: 0.6756304502487183
With Learning rate = 0.002 With batch = 13 Mean Absolute Error test: 0.7896019220352173
With Learning rate = 0.002 With batch = 13 Mean Squared Error test: 1.1196738481521606
With Learning

C:\Users\matte\AppData\Roaming\Python\Python313\site-packages\keras\src\optimizers\base_optimizer.py:86: UserWarning: Argument `decay` is no longer supported and will be ignored.
  warnings.warn(


8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step 
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step 
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
Model indice : 24
With Learning rate = 0.002 With batch = 16 R²_train = 0.8440240144729614
With Learning rate = 0.002 With batch = 16 mse_train = 0.49198166131973264

With Learning rate = 0.002 With batch = 16 R²_val = 0.6779706954956055
With Learning rate = 0.002 With batch = 16 mse_val = 1.0167040705680848

3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step 
With Learning rate = 0.002 With batch = 16 R-squared on Test Set: 0.7387703657150269
With Learning rate = 0.002 With batch = 16 Mean Absolute Error test: 0.7693881988525391
With Learning rate = 0.002 With batch = 16 Mean Squared Error test: 0.9017244577407837
With Learni

In [ ]:
import joblib
import os

os.makedirs('saved_models', exist_ok=True)

ann_rf.save(f'saved_models/ann_lr{lr}_batch{batch}.keras')
joblib.dump(scaler, 'saved_models/scaler_rf.pkl')

In [ ]:
import joblib
import os

os.makedirs('saved_models', exist_ok=True)

lr            = 0.002
batch         = 16
k_folds_train = 5
l2_strength   = 0.01

results_per_seed = {}

for seed in seeds:
    print(f"\n{'='*50}")
    print(f"Entraînement avec seed = {seed}")
    print(f"{'='*50}")

    X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(
        X, y, test_size=0.2, random_state=seed
    )
    scaler_s = StandardScaler()
    X_train_scaled_s = pd.DataFrame(scaler_s.fit_transform(X_train_s), columns=X_train_s.columns)
    X_test_scaled_s  = pd.DataFrame(scaler_s.transform(X_test_s),      columns=X_test_s.columns)

    kf_s = KFold(n_splits=k_folds_train, shuffle=True, random_state=seed)

    tf.random.set_seed(seed)
    np.random.seed(seed)

    model_s = Sequential([
        Dense(32, activation='relu', kernel_initializer=GlorotNormal(seed=seed)),
        Dense(16, activation='relu', kernel_initializer=GlorotNormal(seed=seed)),
        Dropout(0.2),
        Dense(1,  activation='linear', kernel_regularizer=l2(l2_strength))
    ])
    model_s.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr, decay=0.01),
        loss='mse', metrics=['mae']
    )

    r2_train_folds, r2_val_folds = [], []

    for train_idx, val_idx in kf_s.split(X_train_scaled_s):
        X_tr, X_vl = X_train_scaled_s.iloc[train_idx], X_train_scaled_s.iloc[val_idx]
        y_tr, y_vl = y_train_s.iloc[train_idx],        y_train_s.iloc[val_idx]

        model_s.fit(X_tr, y_tr, epochs=100, batch_size=batch, verbose=0,
                    callbacks=[EarlyStopping(patience=20, restore_best_weights=True)])

        r2_train_folds.append(r2_score(y_tr, model_s.predict(X_tr, verbose=0)))
        r2_val_folds.append(  r2_score(y_vl, model_s.predict(X_vl, verbose=0)))

    y_pred_test_s = model_s.predict(X_test_scaled_s, verbose=0).flatten()
    r2_test  = r2_score(y_test_s, y_pred_test_s)
    mae_test = mean_absolute_error(y_test_s, y_pred_test_s)
    mse_test = mean_squared_error(y_test_s, y_pred_test_s)

    print(f"R² train (moy folds) : {np.mean(r2_train_folds):.4f}")
    print(f"R² val   (moy folds) : {np.mean(r2_val_folds):.4f}")
    print(f"R² test              : {r2_test:.4f}")
    print(f"MAE test             : {mae_test:.4f}")
    print(f"RMSE test            : {np.sqrt(mse_test):.4f}")

    results_per_seed[seed] = {
        "r2_train":  np.mean(r2_train_folds),
        "r2_val":    np.mean(r2_val_folds),
        "r2_test":   r2_test,
        "mae_test":  mae_test,
        "rmse_test": np.sqrt(mse_test),
    }

    model_s.save(f'saved_models/ann_seed_{seed}.keras')
    joblib.dump(scaler_s, f'saved_models/scaler_seed_{seed}.pkl')
    print(f"  -> saved_models/ann_seed_{seed}.keras  +  scaler_seed_{seed}.pkl")

print("\n\n=== Récapitulatif ===")
for s, m in results_per_seed.items():
    print(f"seed={s:5d}  R²_test={m['r2_test']:.4f}  MAE={m['mae_test']:.4f}  RMSE={m['rmse_test']:.4f}")